# CX Routing Engine - Jupyter Lab Deployment
This notebook allows you to run the entire Multi-Agent system (vLLM Server + FastAPI Backend + React Frontend) directly within a single Jupyter Lab instance, and uses `ngrok` to expose a public URL so you can view the UI in your browser without worrying about Jupyter port-forwarding issues.

In [ ]:
# 1. Install Dependencies
!pip install -r requirements.txt
!pip install pyngrok vllm langchain-openai

In [ ]:
import subprocess
import time
import os

# 2. Start vLLM Server in the background
print('Starting vLLM Server on Port 8000...')
vllm_process = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server', 
    '--model', 'Qwen/Qwen2.5-14B-Instruct', 
    '--port', '8000', 
    '--tensor-parallel-size', '1'
], stdout=open('vllm_server.log', 'w'), stderr=subprocess.STDOUT)

print('Waiting 60 seconds for 14B model to load into AMD VRAM...')
time.sleep(60)

In [ ]:
import re

# 3. Modify agent.py to use the high-speed vLLM API instead of native transformers
with open('backend/agent.py', 'r') as f:
    agent_code = f.read()

vllm_get_model = """async def get_chat_model():
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(
        model="Qwen/Qwen2.5-14B-Instruct",
        openai_api_base="http://localhost:8000/v1",
        openai_api_key="EMPTY",
        temperature=0.1,
        max_tokens=512
    )
"""

# Replace the native get_chat_model with the vLLM one
agent_code = re.sub(r"async def get_chat_model\(\):.*?return _chat_model", vllm_get_model, agent_code, flags=re.DOTALL)

with open('backend/agent.py', 'w') as f:
    f.write(agent_code)

print('Updated agent.py to point to vLLM server!')

In [ ]:
# 4. Start the FastAPI Backend & Frontend UI
print('Starting FastAPI Application...')
backend_process = subprocess.Popen([
    'uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8001'
], stdout=open('backend_server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)

In [ ]:
# 5. Expose the UI securely via Ngrok
from pyngrok import ngrok

# IMPORTANT: If you want session persistence, place your personal ngrok authtoken below:
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

public_url = ngrok.connect(8001)
print("=" * 60)
print(f"🚀 CX Routing Engine is LIVE at: {public_url}")
print("=" * 60)
print("Click the link above to view your UI. Your frontend and backend are perfectly routed through Jupyter!")